# ARIMA models and forecasting: lab notebook

**Statistical Modelling and Forecasting, ARIMA and vector autoregression**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

### 1. Identify: log real GDP (quarterly, 1959-2009)

In [ ]:
# 1. Identify: log real GDP (quarterly, 1959-2009)
d = sm.datasets.macrodata.load_pandas().data
d.index = pd.period_range("1959Q1", periods=len(d), freq="Q")
y = np.log(d["realgdp"])
print("ADF on the level: p =", round(adfuller(y, regression="ct")[1], 3), "-> unit root not rejected")
dy = y.diff().dropna()
print("ADF on the first difference: p =", round(adfuller(dy)[1], 4), "-> d = 1")
print("ACF of the difference (lags 1-6): ", np.round(acf(dy, nlags=6)[1:], 2))
print("PACF of the difference (lags 1-6):", np.round(pacf(dy, nlags=6)[1:], 2))

### 2. Estimate several candidates on the first 190 quarters and compare

In [ ]:
# 2. Estimate several candidates on the first 190 quarters and compare
train, test = y.iloc[:190], y.iloc[190:]
rows = []
for order in [(0, 1, 0), (1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 0), (2, 1, 1)]:
    r = ARIMA(train, order=order, trend="t").fit()
    rows.append((order, round(r.aic, 1), round(r.bic, 1)))
print(pd.DataFrame(rows, columns=["order", "AIC", "BIC"]).sort_values("AIC").to_string(index=False))

### 3. Diagnose the chosen model

In [ ]:
# 3. Diagnose the chosen model
best = ARIMA(train, order=(1, 1, 1), trend="t").fit()
print(best.summary().tables[1])
lb = acorr_ljungbox(best.resid[1:], lags=[8], model_df=2)
print("Ljung-Box p-value (8 lags):", round(float(lb["lb_pvalue"].iloc[0]), 3), "-> residuals look like white noise" if lb["lb_pvalue"].iloc[0] > 0.05 else "-> structure left")

### 4. Forecast the held-out quarters with 95% intervals and compare with the naive forecast

In [ ]:
# 4. Forecast the held-out quarters with 95% intervals and compare with the naive forecast
fc = best.get_forecast(len(test))
ci = fc.conf_int(alpha=0.05)
mean = fc.predicted_mean
print("first 4 forecasts vs actual (log GDP):")
print(pd.DataFrame({"forecast": mean.iloc[:4], "lower": ci.iloc[:4, 0], "upper": ci.iloc[:4, 1], "actual": test.iloc[:4]}).round(4).to_string())
naive = np.repeat(train.iloc[-1], len(test))
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print("(the 2008-09 recession is in the test set; the drift keeps forecasting growth, so the naive forecast wins here)")
print("RMSE ARIMA(1,1,1):", round(rmse(test.values - mean.values), 4), "| RMSE naive:", round(rmse(test.values - naive), 4))
print("width of the 95% interval at h = 1, 4, 12:", np.round((ci.iloc[:, 1] - ci.iloc[:, 0]).iloc[[0, 3, 11]].values, 3), "(keeps widening because d = 1)")

### 5. The numerical problem

In [ ]:
# 5. The numerical problem
phi, yT, yT1 = 0.5, 100, 96
f1 = yT + phi * (yT - yT1)
print("ARIMA(1,1,0) forecasts:", f1, f1 + phi ** 2 * (yT - yT1))
th, sg = -0.4, 2
print("ARIMA(0,1,1) 3-step standard error:", round(sg * np.sqrt(1 + 2 * (1 + th) ** 2), 3))

## Expected output

Running all cells above should print:

```
ADF on the level: p = 0.389 -> unit root not rejected
ADF on the first difference: p = 0.0 -> d = 1
ACF of the difference (lags 1-6):  [ 0.3   0.24  0.09  0.08 -0.05 -0.04]
PACF of the difference (lags 1-6): [ 0.3   0.16 -0.02  0.02 -0.1  -0.02]
    order     AIC     BIC
(2, 1, 0) -1270.1 -1257.1
(1, 1, 1) -1268.6 -1255.6
(2, 1, 1) -1268.4 -1252.2
(1, 1, 0) -1267.7 -1258.0
(0, 1, 1) -1264.5 -1254.8
(0, 1, 0) -1257.8 -1251.4
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
x1             0.0083      0.001      8.556      0.000       0.006       0.010
ar.L1          0.6357      0.179      3.543      0.000       0.284       0.987
ma.L1         -0.4136      0.210     -1.966      0.049      -0.826      -0.001
sigma2      6.815e-05   5.56e-06     12.247      0.000    5.72e-05    7.91e-05
==============================================================================
Ljung-Box p-value (8 lags): 0.406 -> residuals look like white noise
first 4 forecasts vs actual (log GDP):
        forecast   lower   upper  actual
2006Q3    9.4774  9.4612  9.4935  9.4701
2006Q4    9.4852  9.4596  9.5107  9.4774
2007Q1    9.4932  9.4594  9.5269  9.4804
2007Q2    9.5013  9.4601  9.5424  9.4883
(the 2008-09 recession is in the test set; the drift keeps forecasting growth, so the naive forecast wins here)
RMSE ARIMA(1,1,1): 0.0545 | RMSE naive: 0.0204
width of the 95% interval at h = 1, 4, 12: [0.032 0.082 0.166] (keeps widening because d = 1)
ARIMA(1,1,0) forecasts: 102.0 103.0
ARIMA(0,1,1) 3-step standard error: 2.623
```

## Practice exercises

Try these in the cells below.

1. Take the "ARIMA(1,1,1)" series. Fit (0,1,0), then (1,1,0), then (1,1,1). Watch AIC and the residual ACF.
2. Set d = 0 on this trending series. What happens to the forecast and its interval compared with d = 1?
3. Increase the forecast horizon to 40. Compare how the interval width grows for d = 0 and d = 1.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
